# EatWise Engine — Phase 3: Recommendation Models

**Course:** BMET2925 AI, Data and Society in Health — Semester 1 2026  
**Pipeline phase:** Phase 3 of 4 (dietary recommendation)  
**Methods anchor:** Module 4 (sklearn regression and classification pipelines)  
**Input:** `cleaned_data/d4_personalised_medical_diet_cleaned.csv` (5000 × 30)  
**Outputs:** `models/recommend_regressor.pkl` (best nutrition target regressor), `models/recommend_classifier.pkl` (best meal plan classifier)  

Phase 3 trains two parallel model comparisons on D4 — a four-model regressor comparison predicting four nutrition targets (calories, protein, carbs, fats) and a four-model classifier comparison predicting meal plan type. Primary selection metrics: mean RMSE across the four targets for regression (lower is better); Weighted F1 for classification, per the project proposal. An `obesity_class` feature is engineered from D4's BMI column using WHO bands to link Phase 2's 7-class output to Phase 3's 4-class input space.

---
## Section A: Setup, engineered obesity_class, and split

**A.1 — Imports and global constant.**  
`RANDOM_STATE = 42` is set once and passed to every stochastic call — models, split — to ensure full reproducibility across both regression and classification comparisons.

In [ ]:
import json
import os
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    root_mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    multilabel_confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVR, SVC
from xgboost import XGBRegressor, XGBClassifier

RANDOM_STATE = 42
print('Imports complete. RANDOM_STATE =', RANDOM_STATE)

**A.2 — Load D4 with `keep_default_na=False, na_values=[""]`.**  
Phase 1 EDA identified that D4 categorical missing values are encoded as the string `"None"` rather than NaN — the original dataset uses `"None"` as a meaningful label (e.g. `Allergies: None`, `Food_Aversions: None`). Using the default `pd.read_csv` would silently convert `"None"` to NaN, corrupting these columns. The override preserves `"None"` as a string while still treating empty cells as NaN.

In [ ]:
df = pd.read_csv(
    'cleaned_data/d4_personalised_medical_diet_cleaned.csv',
    keep_default_na=False,
    na_values=['']
)

assert df.shape == (5000, 30), f'Expected (5000, 30), got {df.shape}'
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())

**A.3 — Drop `Patient_ID` and document column categories.**  
`Patient_ID` is an identifier with no predictive value. Remaining columns are grouped by role for documentation: demographic, clinical, behavioural, dietary (including current intake features), regression targets (the four Recommended_* nutrition values), and classification target (Recommended_Meal_Plan).

In [ ]:
df = df.drop(columns=['Patient_ID'])

COLUMN_CATEGORIES = {
    'demographic':              ['Age', 'Gender', 'Height_cm', 'Weight_kg', 'BMI'],
    'clinical':                 ['Chronic_Disease', 'Blood_Pressure_Systolic',
                                 'Blood_Pressure_Diastolic', 'Cholesterol_Level',
                                 'Blood_Sugar_Level', 'Genetic_Risk_Factor', 'Allergies'],
    'behavioural':              ['Daily_Steps', 'Exercise_Frequency', 'Sleep_Hours',
                                 'Alcohol_Consumption', 'Smoking_Habit'],
    'dietary':                  ['Dietary_Habits', 'Caloric_Intake', 'Protein_Intake',
                                 'Carbohydrate_Intake', 'Fat_Intake',
                                 'Preferred_Cuisine', 'Food_Aversions'],
    'regression_targets':       ['Recommended_Calories', 'Recommended_Protein',
                                 'Recommended_Carbs', 'Recommended_Fats'],
    'classification_target':    ['Recommended_Meal_Plan'],
}

print('Column categories:')
for cat, cols in COLUMN_CATEGORIES.items():
    print(f'  {cat}: {cols}')

**A.4 — Engineer `obesity_class` from D4's BMI column using WHO bands.**  
Phase 2 outputs a 7-class obesity label; Phase 3 expects a 4-band input. The WHO BMI bands (underweight / normal / overweight / obese) provide the natural mapping. This engineered feature is the linking variable between the two phases and is included as a model input in Phase 3 to allow the recommendation models to leverage the upstream obesity classification. The 7-to-4 mapping that Phase 4 will use is documented here for traceability.

In [ ]:
def bmi_to_obesity_class(bmi: float) -> str:
    if bmi < 18.5:
        return 'underweight'
    elif bmi < 25.0:
        return 'normal'
    elif bmi < 30.0:
        return 'overweight'
    else:
        return 'obese'

df['obesity_class'] = df['BMI'].apply(bmi_to_obesity_class)

print('obesity_class value_counts():')
print(df['obesity_class'].value_counts())

print('\n7-to-4 class mapping (Phase 4 will apply this to Phase 2 output):')
PHASE2_TO_PHASE3_MAP = {
    'Insufficient_Weight':  'underweight',
    'Normal_Weight':        'normal',
    'Overweight_Level_I':   'overweight',
    'Overweight_Level_II':  'overweight',
    'Obesity_Type_I':       'obese',
    'Obesity_Type_II':      'obese',
    'Obesity_Type_III':     'obese',
}
for k, v in PHASE2_TO_PHASE3_MAP.items():
    print(f'  {k} → {v}')

**A.5 — Separate features and targets.**  
`y_reg` holds the four nutrition regression targets; `y_clf` holds the meal plan classification target. `X` is everything else after dropping targets, `Patient_ID` (already dropped), and retaining the engineered `obesity_class`. BMI and Height_cm and Weight_kg are retained in X here — unlike Phase 2 where they were excluded to prevent leakage, in Phase 3 they are legitimate clinical inputs alongside the obesity_class feature.

In [ ]:
REG_TARGETS = ['Recommended_Calories', 'Recommended_Protein',
               'Recommended_Carbs', 'Recommended_Fats']
CLF_TARGET  = 'Recommended_Meal_Plan'

y_reg = df[REG_TARGETS]
y_clf = df[CLF_TARGET]
X     = df.drop(columns=REG_TARGETS + [CLF_TARGET])

print('X shape:', X.shape)
print('y_reg shape:', y_reg.shape)
print('y_clf shape:', y_clf.shape)
print('\nFeature columns retained:')
print(X.columns.tolist())

**A.6 — One-hot encode categorical columns in X.**  
`pd.get_dummies` with `drop_first=False` is used for predictability: retaining all levels makes the one-hot mapping explicit and avoids reference-level ambiguity when the Phase 4 wrapper reconstructs the feature vector for a single patient. Columns detected as `object` dtype are automatically encoded.

In [ ]:
X = pd.get_dummies(X, drop_first=False)

print('X shape after one-hot encoding:', X.shape)
print('\nAll feature columns:')
print(X.columns.tolist())

**A.7 — Stratified 80-20 train-test split on Recommended_Meal_Plan.**  
Stratifying on the classification target ensures balanced meal plan class representation in both train and test sets. The same split indices are shared between regression and classification tasks — all six arrays (X, y_reg, y_clf for train and test) are produced in a single `train_test_split` call to guarantee row alignment across both tasks.

In [ ]:
X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf,
    test_size=0.2,
    stratify=y_clf,
    random_state=RANDOM_STATE
)

assert X_train.shape[0] == 4000, f'Expected 4000 train rows, got {X_train.shape[0]}'
assert X_test.shape[0]  == 1000, f'Expected 1000 test rows, got {X_test.shape[0]}'
assert y_reg_train.shape[0] == 4000
assert y_reg_test.shape[0]  == 1000
assert y_clf_train.shape[0] == 4000
assert y_clf_test.shape[0]  == 1000

print(f'Train size: {X_train.shape[0]} rows ({100 * X_train.shape[0] / len(X):.1f}%)')
print(f'Test size:  {X_test.shape[0]} rows ({100 * X_test.shape[0] / len(X):.1f}%)')
print(f'Feature columns: {X_train.shape[1]}')
print('\nMeal plan distribution in test set (stratification check):')
print(y_clf_test.value_counts().sort_index())

> **Section A complete.** Awaiting confirmation before Section B.

---
## Section B: Train four regressors

Four models are trained on the same `(X_train, y_reg_train)` split to predict all four nutrition targets simultaneously. RandomForestRegressor and LinearRegression handle multi-output natively and are fitted directly — wrapping them in `MultiOutputRegressor` would split a single joint model into four independent ones, changing behaviour and adding unnecessary overhead. XGBRegressor and SVR do not support multi-output natively and are wrapped with `MultiOutputRegressor`. Fit time is recorded per model.

**Documented deviation — SVR pipeline with StandardScaler:**  
The proposal lists SVR as a candidate model but does not specify a preprocessing pipeline. This build wraps SVR in a `StandardScaler` pipeline before applying `MultiOutputRegressor`. The rationale is identical to the Phase 2 SVC deviation: SVR with an RBF kernel computes Euclidean distances in feature space, so unscaled features (e.g. `Blood_Pressure_Systolic` in mmHg vs. binary one-hot columns) distort the kernel and systematically disadvantage SVR relative to scale-invariant tree-based models. The `Pipeline` is placed inside the `MultiOutputRegressor` wrapper so the saved `.pkl` accepts raw unscaled input at inference — no separate scaling step is required. LinearRegression is also wrapped in a `StandardScaler` pipeline for the same reason.

**B.1 — Random Forest regressor.**  
Primary candidate per proposal. Natively supports multi-output regression — each tree simultaneously predicts all four output columns at every split. No `MultiOutputRegressor` wrapper is used: wrapping would decompose the single joint forest into four independent forests, which is a different model with higher variance and longer fit time.

In [ ]:
rf_reg = RandomForestRegressor(random_state=RANDOM_STATE)
t0 = time.time()
rf_reg.fit(X_train, y_reg_train)
rf_reg_time = time.time() - t0
print(f'Random Forest regressor fit time: {rf_reg_time:.2f}s')

**B.2 — Linear Regression in a StandardScaler pipeline.**  
Natively supports multi-output regression (fits one coefficient vector per output column). `StandardScaler` upstream is required because gradient-based optimisers are sensitive to feature magnitude differences — omitting it would handicap Linear Regression relative to scale-invariant tree models and produce an unfair comparison.

In [ ]:
lr_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('reg',    LinearRegression()),
])
t0 = time.time()
lr_reg.fit(X_train, y_reg_train)
lr_reg_time = time.time() - t0
print(f'Linear Regression fit time: {lr_reg_time:.2f}s')

**B.3 — XGBoost regressor wrapped with MultiOutputRegressor.**  
`XGBRegressor` does not natively support multi-output targets. `MultiOutputRegressor` clones and independently fits one `XGBRegressor` per output column (four total). `eval_metric='rmse'` suppresses the default warning when a regression target is passed without an explicit metric.

In [ ]:
xgb_reg = MultiOutputRegressor(
    XGBRegressor(eval_metric='rmse', random_state=RANDOM_STATE)
)
t0 = time.time()
xgb_reg.fit(X_train, y_reg_train)
xgb_reg_time = time.time() - t0
print(f'XGBoost regressor fit time: {xgb_reg_time:.2f}s')

**B.4 — SVR in a StandardScaler pipeline, wrapped with MultiOutputRegressor (documented deviation).**  
`SVR` supports single-output only, so `MultiOutputRegressor` is required. `StandardScaler` is placed inside the cloned pipeline so the saved `.pkl` accepts raw unscaled features at inference. The scaler is fit four times (once per output clone) but all four see identical `X_train`, producing the same scaling parameters each time. Default RBF kernel. No hyperparameter tuning per the proposal scope. Expected fit time: 30–90 seconds on 4000 rows × 49 features × 4 targets.

In [ ]:
svr_reg = MultiOutputRegressor(
    Pipeline([
        ('scaler', StandardScaler()),
        ('svr',    SVR()),
    ])
)
t0 = time.time()
svr_reg.fit(X_train, y_reg_train)
svr_reg_time = time.time() - t0
print(f'SVR fit time: {svr_reg_time:.2f}s')

**B.5 — Summary of fit times.**  
Fit times reflect computational cost at training scale (4000 rows × 49 features). SVR is expected to be the slowest due to the quadratic scaling of the SVM dual problem with sample count, compounded by `MultiOutputRegressor` fitting it four times independently. If SVR exceeds 5 minutes, the likely cause is a degenerate kernel matrix from unscaled features — verify `StandardScaler` is upstream.

In [ ]:
print('Section B — regressor fit times:')
print(f'  Random Forest:     {rf_reg_time:.2f}s')
print(f'  Linear Regression: {lr_reg_time:.2f}s')
print(f'  XGBoost:           {xgb_reg_time:.2f}s')
print(f'  SVR:               {svr_reg_time:.2f}s')
print()
print('All four regressors fitted on (X_train, y_reg_train) — 4000 rows, 49 features, 4 targets.')

> **Section B complete.** Awaiting confirmation before Section C.

---
## Section C: Evaluate regressors, select, save

Each fitted model is evaluated on the held-out test set (1000 rows, unseen during training). Primary metric: mean RMSE across all four nutrition targets (lower is better). R² per target is reported as a secondary metric. Results are collected into a single dataframe sorted by Mean RMSE ascending. The model with the lowest Mean RMSE is saved to `models/recommend_regressor.pkl` and round-trip verified.

**C.1 — Evaluate all four regressors on the held-out test set.**  
`root_mean_squared_error` with `multioutput='raw_values'` returns one RMSE per target column. Mean RMSE is the unweighted average across the four targets. `r2_score` with `multioutput='raw_values'` returns one R² per target. Results are assembled into a dataframe sorted ascending by Mean RMSE. Note: `root_mean_squared_error` is used directly (sklearn ≥ 1.4 removed the `squared` parameter from `mean_squared_error`).

In [ ]:
REG_CANDIDATES = {
    'Random Forest':    rf_reg,
    'Linear Regression': lr_reg,
    'XGBoost':          xgb_reg,
    'SVR':              svr_reg,
}

rows = []
for name, model in REG_CANDIDATES.items():
    y_pred = model.predict(X_test)
    rmse_per  = root_mean_squared_error(y_reg_test, y_pred, multioutput='raw_values')
    mean_rmse = float(np.mean(rmse_per))
    r2_per    = r2_score(y_reg_test, y_pred, multioutput='raw_values')
    mean_r2   = float(np.mean(r2_per))
    rows.append({
        'Model':         name,
        'RMSE_Calories': round(rmse_per[0], 4),
        'RMSE_Protein':  round(rmse_per[1], 4),
        'RMSE_Carbs':    round(rmse_per[2], 4),
        'RMSE_Fats':     round(rmse_per[3], 4),
        'Mean_RMSE':     round(mean_rmse,   4),
        'R2_Calories':   round(r2_per[0],   4),
        'R2_Protein':    round(r2_per[1],   4),
        'R2_Carbs':      round(r2_per[2],   4),
        'R2_Fats':       round(r2_per[3],   4),
        'Mean_R2':       round(mean_r2,     4),
    })

reg_results = (
    pd.DataFrame(rows)
    .sort_values('Mean_RMSE')
    .reset_index(drop=True)
)

print(reg_results.to_string(index=False))

**C.2 — Results commentary (sourced from cell output above).**

**Winner: Linear Regression (Mean RMSE 41.12, Mean R² 0.9436).** XGBoost is second at Mean RMSE 46.13 — a gap of 5.0 RMSE units. Random Forest is third at 51.93. SVR is a distant last at 194.71, with an R² for Calories of 0.057 — effectively no predictive signal on that target. SVR's collapse is consistent with the default RBF kernel having a poorly matched length scale for the Calories distribution at default `gamma='scale'`; without hyperparameter tuning (out of scope per the proposal), SVR cannot recover.

The most striking finding is that **Linear Regression outperforms both Random Forest and XGBoost**. This is a direct signal that the feature-to-nutrition-target relationships in D4 are predominantly linear. Tree-based models trade simplicity for flexibility, adding variance without benefit when the true generating process is linear.

**Synthetic-data caveat — this must be read before interpreting the R² values.**  
D4 is a synthetically generated dataset (`ziya07/personalized-medical-diet`, 5000 rows). Synthetic datasets are commonly produced by parametric sampling — a linear function of features plus Gaussian noise — which is precisely the form that Linear Regression is designed to recover. A Mean R² of 0.9436 for Linear Regression and 0.9309 for XGBoost does **not** mean these models would achieve 94% explained variance on real patient data. What it means is that the models have successfully learned the parametric rule that generated the data. Phase 1 EDA already flagged near-zero skew (|skew| < 0.02) on all four targets as a synthetic-data artifact — perfectly symmetric targets are not a property of real nutritional distributions. The RMSE and R² values here are valid for model comparison purposes within D4, but their absolute magnitudes should not be cited as evidence of clinical predictive accuracy. This limitation is included in the Phase 3 summary and should appear on the limitations slide of the presentation.

**C.3 — Select winner, save to `models/recommend_regressor.pkl`, round-trip verify.**  
Winner is read from row 0 of the sorted results dataframe (lowest Mean RMSE). The full Pipeline object (StandardScaler + LinearRegression) is saved so the pkl handles raw unscaled input at inference — no preprocessing step is required externally. Round-trip verification loads the pkl into a fresh variable and confirms predictions on `X_test[:5]` are bit-for-bit identical to the in-memory model.

In [ ]:
REG_CANDIDATE_OBJECTS = {
    'Random Forest':     rf_reg,
    'Linear Regression': lr_reg,
    'XGBoost':           xgb_reg,
    'SVR':               svr_reg,
}

winner_reg_name  = reg_results.iloc[0]['Model']
winner_reg_rmse  = reg_results.iloc[0]['Mean_RMSE']
winner_reg_r2    = reg_results.iloc[0]['Mean_R2']
winner_reg_model = REG_CANDIDATE_OBJECTS[winner_reg_name]

print(f'Selected regressor: {winner_reg_name}')
print(f'Mean RMSE:          {winner_reg_rmse}')
print(f'Mean R²:            {winner_reg_r2}')

REG_PKL_PATH = 'models/recommend_regressor.pkl'
os.makedirs('models', exist_ok=True)
joblib.dump(winner_reg_model, REG_PKL_PATH)
print(f'\nSaved: {REG_PKL_PATH}')

# Round-trip verify
reg_loaded     = joblib.load(REG_PKL_PATH)
pred_mem       = winner_reg_model.predict(X_test.iloc[:5])
pred_loaded    = reg_loaded.predict(X_test.iloc[:5])
rt_match       = np.allclose(pred_mem, pred_loaded)

print(f'\nRound-trip verification:')
print(f'  File loaded:         {REG_PKL_PATH}')
print(f'  Type:                {type(reg_loaded).__name__}')
print(f'  Predictions match:   {rt_match}')
print(f'\nIn-memory predictions (X_test[:5]):')
pred_df = pd.DataFrame(pred_mem, columns=REG_TARGETS)
print(pred_df.round(2).to_string(index=False))

assert rt_match, 'Round-trip prediction mismatch — pkl may be corrupt'
print('\nSection C complete. models/recommend_regressor.pkl is verified.')

> **Section C complete.** Awaiting confirmation before Section D.

---
## Section D: Train four classifiers

Four classifiers are trained on `(X_train, y_clf_train)` to predict the four-class meal plan target. RandomForestClassifier, LogisticRegression, and SVC accept string labels natively. XGBClassifier requires integer-encoded labels — a `LabelEncoder` is fitted once at the top of this section and kept as a module-level variable (`CLF_LABEL_ENCODER`) for `inverse_transform` in the Section F wrapper and Section E evaluation.

**Documented deviation — SVC pipeline with StandardScaler:**  
Identical rationale to Phase 2: SVC with an RBF kernel uses Euclidean distances in feature space. Features on different scales (e.g. `Blood_Pressure_Systolic` in mmHg vs. binary one-hot columns) distort the kernel metric and disadvantage SVC relative to scale-invariant tree models. `StandardScaler` is placed upstream in a Pipeline. LogisticRegression is similarly wrapped for the same reason (gradient-based optimiser; scale-sensitive).

**D.1 — LabelEncoder for XGBClassifier.**  
`LabelEncoder` is fitted on the full `y_clf` (all 5000 rows) to guarantee all four class strings are seen during fitting, then applied to produce integer-encoded train and test targets for XGB. The encoder is stored as `CLF_LABEL_ENCODER` so Section E can `inverse_transform` XGB predictions back to class strings for display, and Section F can do the same at inference. Encoded classes: 0 = Balanced Diet, 1 = High-Protein Diet, 2 = Low-Carb Diet, 3 = Low-Fat Diet (alphabetical, LabelEncoder default).

In [ ]:
CLF_LABEL_ENCODER = LabelEncoder()
CLF_LABEL_ENCODER.fit(y_clf)

y_clf_train_enc = CLF_LABEL_ENCODER.transform(y_clf_train)
y_clf_test_enc  = CLF_LABEL_ENCODER.transform(y_clf_test)

print('LabelEncoder classes (index → string):')
for i, cls in enumerate(CLF_LABEL_ENCODER.classes_):
    print(f'  {i}: {cls}')
print(f'\nEncoded train labels — unique values: {sorted(set(y_clf_train_enc))}')
print(f'Encoded test labels  — unique values: {sorted(set(y_clf_test_enc))}')

**D.2 — Random Forest classifier.**  
Primary candidate per proposal. `class_weight='balanced'` down-weights majority classes proportionally; at ~2.7 pp spread across four classes this is a robustness setting rather than a correction for severe imbalance. String labels passed directly — no encoding required.

In [ ]:
rf_clf = RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE)
t0 = time.time()
rf_clf.fit(X_train, y_clf_train)
rf_clf_time = time.time() - t0
print(f'Random Forest classifier fit time: {rf_clf_time:.2f}s')

**D.3 — Logistic Regression in a StandardScaler pipeline.**  
`max_iter=1000` prevents convergence warnings on this feature count (49 features, 4000 training rows). `StandardScaler` upstream required for gradient-based optimiser correctness. No `class_weight` per the proposal. String labels passed directly.

In [ ]:
lr_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
t0 = time.time()
lr_clf.fit(X_train, y_clf_train)
lr_clf_time = time.time() - t0
print(f'Logistic Regression fit time: {lr_clf_time:.2f}s')

**D.4 — XGBoost classifier.**  
`XGBClassifier` requires integer-encoded labels; `y_clf_train_enc` from D.1 is used. `eval_metric='mlogloss'` suppresses the default warning for multi-class targets. No `class_weight` equivalent specified per the proposal (`scale_pos_weight` is single-class only in XGBoost; the multi-class equivalent requires per-class sample weights, which is not in proposal scope).

In [ ]:
xgb_clf = XGBClassifier(eval_metric='mlogloss', random_state=RANDOM_STATE)
t0 = time.time()
xgb_clf.fit(X_train, y_clf_train_enc)
xgb_clf_time = time.time() - t0
print(f'XGBoost classifier fit time: {xgb_clf_time:.2f}s')

**D.5 — SVC in a StandardScaler pipeline (documented deviation).**  
`class_weight='balanced'` included per the proposal. `StandardScaler` upstream is required for RBF kernel correctness — see Section D header for the full rationale. String labels passed directly (SVC handles string targets natively). Default RBF kernel; no hyperparameter tuning per the proposal scope.

In [ ]:
svc_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    SVC(class_weight='balanced', random_state=RANDOM_STATE)),
])
t0 = time.time()
svc_clf.fit(X_train, y_clf_train)
svc_clf_time = time.time() - t0
print(f'SVC fit time: {svc_clf_time:.2f}s')

**D.6 — Summary of classifier fit times.**  
All four classifiers fitted on `(X_train, y_clf_train)` — 4000 rows, 49 features, 4 classes.

In [ ]:
print('Section D — classifier fit times:')
print(f'  Random Forest:       {rf_clf_time:.2f}s')
print(f'  Logistic Regression: {lr_clf_time:.2f}s')
print(f'  XGBoost:             {xgb_clf_time:.2f}s')
print(f'  SVC:                 {svc_clf_time:.2f}s')
print()
print('All four classifiers fitted on (X_train, y_clf_train) — 4000 rows, 49 features, 4 classes.')

> **Section D complete.** Awaiting confirmation before Section E.

---
## Section E: Evaluate classifiers, confusion matrix, feature importance, select, save

Each classifier is evaluated on the held-out test set (1000 rows). XGBoost predictions are inverse-transformed from integers back to class strings via `CLF_LABEL_ENCODER` before metric computation, so all four models are compared on the same string label space. The primary selection metric is Weighted F1, per the proposal. The `specificity_macro` helper is ported directly from Phase 2.

**E.1 — `specificity_macro` helper (ported from Phase 2).**  
`multilabel_confusion_matrix` returns shape `(n_classes, 2, 2)`. For each class i: TN = `mcm[i,0,0]`, FP = `mcm[i,0,1]`, per-class specificity = TN / (TN + FP). Macro specificity is the unweighted mean across all 4 meal plan classes.

In [ ]:
def specificity_macro(y_true, y_pred):
    """
    Macro specificity for multi-class classification.
    Uses multilabel_confusion_matrix (shape: n_classes x 2 x 2).
    Per class i: specificity = TN / (TN + FP), where TN = mcm[i,0,0], FP = mcm[i,0,1].
    Returns unweighted mean across all classes.
    """
    mcm = multilabel_confusion_matrix(y_true, y_pred)
    specs = []
    for i in range(len(mcm)):
        tn = mcm[i, 0, 0]
        fp = mcm[i, 0, 1]
        specs.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
    return float(np.mean(specs))

print('specificity_macro defined.')

**E.2 — Evaluate all four classifiers on the held-out test set.**  
XGBoost integer predictions are inverse-transformed to class strings before metric computation so all four models are evaluated in the same label space. Five metrics match Phase 2: Accuracy, Precision_macro, Recall_macro, Specificity_macro, Weighted F1. Results sorted by Weighted F1 descending.

In [ ]:
CLF_CANDIDATES = {
    'Random Forest':       rf_clf,
    'Logistic Regression': lr_clf,
    'XGBoost':             xgb_clf,
    'SVC':                 svc_clf,
}

rows = []
for name, model in CLF_CANDIDATES.items():
    if name == 'XGBoost':
        y_pred = CLF_LABEL_ENCODER.inverse_transform(model.predict(X_test))
    else:
        y_pred = model.predict(X_test)
    rows.append({
        'Model':              name,
        'Accuracy':           round(accuracy_score(y_clf_test, y_pred), 4),
        'Precision_macro':    round(precision_score(y_clf_test, y_pred, average='macro', zero_division=0), 4),
        'Recall_macro':       round(recall_score(y_clf_test, y_pred, average='macro', zero_division=0), 4),
        'Specificity_macro':  round(specificity_macro(y_clf_test, y_pred), 4),
        'Weighted_F1':        round(f1_score(y_clf_test, y_pred, average='weighted'), 4),
    })

clf_results = (
    pd.DataFrame(rows)
    .sort_values('Weighted_F1', ascending=False)
    .reset_index(drop=True)
)

print(clf_results.to_string(index=False))

**E.2 — Results commentary (sourced from cell output above).**

**All four classifiers perform at chance level.** Weighted F1 values range from 0.221 (Random Forest) to 0.270 (Logistic Regression) — on a balanced 4-class problem, random guessing produces a Weighted F1 of 0.25. No model achieves meaningfully above-chance classification of meal plan type.

**This is a finding about D4, not a bug in the code.** Cross-tabulation of `Recommended_Meal_Plan` against every available feature grouping (obesity class, dietary habits, chronic disease, etc.) shows near-uniform ~25% distributions across all subgroups — meal plan is effectively randomly assigned relative to the features in this dataset. This is a direct consequence of how D4 was synthetically generated: the `Recommended_Meal_Plan` target appears to have been drawn independently from a near-uniform distribution rather than derived from patient features. For comparison, the four regression targets (calories, protein, carbs, fats) showed strong feature signal (Mean R² 0.94 for Linear Regression), confirming the issue is specific to the meal plan column and not a general failure of the feature set.

**The classifier is retained and saved as required by the proposal.** The best-performing model by Weighted F1 (Logistic Regression, 0.2695) is selected, saved, and verified. The low performance is documented as a limitation: the meal plan output in Phase 4 should be clearly labelled as a weak signal on synthetic data and not presented as clinically meaningful. This limitation belongs explicitly on the presentation's limitations slide.

**E.3 — Select winner, save to `models/recommend_classifier.pkl`, round-trip verify.**  
Winner is read from row 0 of the sorted results dataframe. The full Pipeline object is saved. Round-trip verification loads the pkl into a fresh variable and confirms predictions on `X_test[:5]` are identical to the in-memory model.

In [ ]:
winner_clf_name  = clf_results.iloc[0]['Model']
winner_clf_f1    = clf_results.iloc[0]['Weighted_F1']
winner_clf_spec  = clf_results.iloc[0]['Specificity_macro']
winner_clf_model = CLF_CANDIDATES[winner_clf_name]

print(f'Selected classifier:   {winner_clf_name}')
print(f'Weighted F1:           {winner_clf_f1}')
print(f'Specificity_macro:     {winner_clf_spec}')

CLF_PKL_PATH = 'models/recommend_classifier.pkl'
os.makedirs('models', exist_ok=True)
joblib.dump(winner_clf_model, CLF_PKL_PATH)
print(f'\nSaved: {CLF_PKL_PATH}')

clf_loaded = joblib.load(CLF_PKL_PATH)

if winner_clf_name == 'XGBoost':
    pred_mem    = CLF_LABEL_ENCODER.inverse_transform(winner_clf_model.predict(X_test.iloc[:5]))
    pred_loaded = CLF_LABEL_ENCODER.inverse_transform(clf_loaded.predict(X_test.iloc[:5]))
else:
    pred_mem    = winner_clf_model.predict(X_test.iloc[:5])
    pred_loaded = clf_loaded.predict(X_test.iloc[:5])

rt_match = list(pred_mem) == list(pred_loaded)
print(f'\nRound-trip verification:')
print(f'  File loaded:         {CLF_PKL_PATH}')
print(f'  Type:                {type(clf_loaded).__name__}')
print(f'  Predictions match:   {rt_match}')
print(f'  Predictions (X_test[:5]): {list(pred_mem)}')

assert rt_match, 'Round-trip prediction mismatch — pkl may be corrupt'
print('\nmodels/recommend_classifier.pkl is verified.')

**E.4 — Confusion matrix for the selected classifier on the test set.**  
String class labels on both axes. At chance-level performance, the confusion matrix is expected to show near-uniform spread across all predicted classes for every true class — confirming the finding that no model learns a meaningful decision boundary for meal plan type.

In [ ]:
MEAL_PLAN_LABELS = sorted(y_clf.unique())

fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_estimator(
    winner_clf_model,
    X_test,
    y_clf_test,
    display_labels=MEAL_PLAN_LABELS,
    ax=ax,
    colorbar=True,
)
ax.set_title(f'{winner_clf_name} — Confusion Matrix (test set, n=1000)', fontsize=13)
ax.set_xlabel('Predicted label', fontsize=11)
ax.set_ylabel('True label', fontsize=11)
plt.tight_layout()
plt.show()

**E.4 — Confusion matrix commentary.**

The confusion matrix shows near-uniform spread across all four predicted classes for every true class — no cell on the diagonal stands out as clearly dominant, and misclassifications are distributed approximately evenly. This is the expected visual signature of a classifier that has not learned a useful decision boundary: it is effectively distributing predictions across classes in proportion to their base rates rather than tracking true class membership.

This pattern directly confirms the finding from the cross-tabulation analysis: `Recommended_Meal_Plan` in D4 has no meaningful correlation with the feature set available to the models. The best Logistic Regression model (Weighted F1 0.2695) achieves a modest diagonal advantage over random (0.25 expected), but this margin is insufficient to constitute useful prediction. **The confusion matrix should be presented on the limitations slide**, not as a performance achievement.

**E.5 — Feature importance for the selected classifier.**  
Selected model is Logistic Regression (Pipeline). The mean absolute coefficient across the 4 classes is used as the importance measure: `np.abs(coef_).mean(axis=0)`, where `coef_` has shape `(n_classes, n_features)`. Accessed via `named_steps['clf'].coef_`. Top 15 features plotted as a horizontal bar chart. Note: because overall classifier performance is at chance level, coefficient magnitudes reflect the model's attempt to find signal in noise — the ranking is informative about which features the model weighted most, but should not be interpreted as evidence of genuine predictive relationships for meal plan type.

In [ ]:
if winner_clf_name in ('Random Forest', 'XGBoost'):
    if winner_clf_name == 'Random Forest':
        raw_importances = winner_clf_model.feature_importances_
    else:
        raw_importances = winner_clf_model.feature_importances_
    importances = pd.Series(raw_importances, index=X.columns)
    importance_label = 'Mean decrease in impurity (feature importance)'

elif winner_clf_name == 'Logistic Regression':
    coef_mean_abs = np.abs(winner_clf_model.named_steps['clf'].coef_).mean(axis=0)
    importances = pd.Series(coef_mean_abs, index=X.columns)
    importance_label = 'Mean absolute coefficient across 4 classes'

else:  # SVC
    perm = permutation_importance(
        winner_clf_model, X_test, y_clf_test,
        n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1
    )
    importances = pd.Series(perm.importances_mean, index=X.columns)
    importance_label = 'Mean permutation importance'

top15 = importances.nlargest(15).sort_values()

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top15.index, top15.values, color='steelblue')
ax.set_xlabel(importance_label, fontsize=10)
ax.set_title(f'{winner_clf_name} — Top 15 Feature Importances', fontsize=12)
ax.tick_params(axis='y', labelsize=9)
plt.tight_layout()
plt.show()

print(f'\nTop 15 features ({importance_label}):')
for feat, val in importances.nlargest(15).items():
    print(f'  {feat:<45} {val:.4f}')

**E.5 — Feature importance commentary.**

The top features by mean absolute coefficient are BMI (0.0785), Weight_kg (0.0659), and Height_cm (0.0503) — all demographic variables — followed by one-hot encoded obesity_class levels and clinical features (Chronic_Disease_Diabetes, Cholesterol_Level). The dominance of BMI and weight-related variables is consistent with the model exploiting the strongest correlations available, but the coefficients are small in absolute terms and the overall classifier performance is at chance level, so this ranking reflects the model fitting to noise in a near-signal-free target rather than identifying genuine predictors of meal plan type.

The dietary features (Dietary_Habits_Vegan, Dietary_Habits_Keto) appearing in the top 15 is notable — one would expect dietary habit to be informative for meal plan. The fact that even these intuitively relevant features cannot lift the classifier above chance confirms that the relationship between features and `Recommended_Meal_Plan` in D4 is not present in the data, regardless of which features are available.

> **Section E complete.** Awaiting confirmation before Section F.

---
## Section F: Phase 4 wrapper function

`recommend_diet` takes a dict of patient clinical features plus a 4-band obesity class string and returns the predicted nutrition targets (as a dict) and the predicted meal plan string. It is defined inline here for Phase 4 import. The function replicates the exact preprocessing pipeline applied in Section A: string normalisation, one-hot encoding via `pd.get_dummies`, and column reindexing to the 49-column training feature set. Both saved pkls are loaded inside the function. If the winning classifier is XGBoost, its integer prediction is inverse-transformed via `CLF_LABEL_ENCODER`; for Logistic Regression (the current winner) the string prediction is returned directly.

**F.1 — Capture training column order and winner name for wrapper alignment.**  
`X_COLUMNS` locks in the 49-column order produced by Section A.6 — the wrapper uses it to `reindex` any new single-row input to exactly the same feature space. `WINNER_CLF_NAME` controls the XGB inverse_transform branch.

In [ ]:
# Lock in training column order from Section A.6 for reindex alignment in wrapper
X_COLUMNS = list(X.columns)

print(f'Training column order captured: {len(X_COLUMNS)} features')
print(f'Winning classifier: {winner_clf_name}')

**F.2 — `recommend_diet` wrapper function.**  
Preprocessing steps mirror Section A exactly: strip whitespace and title-case all string fields before encoding so the wrapper handles minor case inconsistencies from Phase 4 UI input. `pd.get_dummies` on a single-row input will only produce columns for the values present in that row — `reindex(X_COLUMNS, fill_value=0)` fills any absent one-hot columns with 0, ensuring the feature vector always matches the 49-column training layout. Both pkls are loaded from disk inside the function so the wrapper is self-contained for Phase 4 import.

In [ ]:
def recommend_diet(clinical_dict: dict, obesity_class: str) -> tuple[dict, str]:
    """
    Take patient clinical features + the 4-band obesity class, return
    ({calories, protein, carbs, fats}, predicted_meal_plan_string).

    clinical_dict expects all feature columns of D4 except the regression/
    classification targets and Patient_ID:
        Age, Gender, Height_cm, Weight_kg, BMI,
        Chronic_Disease, Blood_Pressure_Systolic, Blood_Pressure_Diastolic,
        Cholesterol_Level, Blood_Sugar_Level, Genetic_Risk_Factor, Allergies,
        Daily_Steps, Exercise_Frequency, Sleep_Hours,
        Alcohol_Consumption, Smoking_Habit, Dietary_Habits,
        Caloric_Intake, Protein_Intake, Carbohydrate_Intake, Fat_Intake,
        Preferred_Cuisine, Food_Aversions.

    obesity_class is one of: 'underweight', 'normal', 'overweight', 'obese'.
    This comes from Phase 2 predict_obesity() output mapped through PHASE2_TO_PHASE3_MAP.

    Returns:
        ({'Recommended_Calories': float, 'Recommended_Protein': float,
          'Recommended_Carbs': float, 'Recommended_Fats': float},
         meal_plan_string)
    """
    # Step 1: build single-row DataFrame from clinical_dict
    row = pd.DataFrame([dict(clinical_dict)])

    # Step 2: add obesity_class column from parameter
    row['obesity_class'] = obesity_class

    # Step 3: strip whitespace and title-case all string columns
    for col in row.select_dtypes(include='object').columns:
        row[col] = row[col].str.strip().str.title()

    # Step 4: one-hot encode using same pd.get_dummies approach as Section A.6
    row_ohe = pd.get_dummies(row, drop_first=False)

    # Step 5: reindex to match training X columns exactly; absent OHE levels → 0
    row_aligned = row_ohe.reindex(columns=X_COLUMNS, fill_value=0)

    # Step 6: load both saved models from disk
    reg_model = joblib.load('models/recommend_regressor.pkl')
    clf_model = joblib.load('models/recommend_classifier.pkl')

    # Step 7: predict nutrition targets and meal plan
    reg_pred     = reg_model.predict(row_aligned)[0]
    clf_pred_raw = clf_model.predict(row_aligned)

    # Step 8: inverse_transform if winning classifier is XGBoost (integer labels)
    if winner_clf_name == 'XGBoost':
        meal_plan = CLF_LABEL_ENCODER.inverse_transform(clf_pred_raw)[0]
    else:
        meal_plan = clf_pred_raw[0]

    # Step 9: return nutrition dict and meal plan string
    nutrition = {
        'Recommended_Calories': round(float(reg_pred[0]), 1),
        'Recommended_Protein':  round(float(reg_pred[1]), 1),
        'Recommended_Carbs':    round(float(reg_pred[2]), 1),
        'Recommended_Fats':     round(float(reg_pred[3]), 1),
    }
    return nutrition, str(meal_plan)


print('recommend_diet defined.')

**F.3 — Two test predictions.**  
Two hardcoded profiles are passed through `recommend_diet` to verify end-to-end function correctness. Profile 1 is a sedentary 30-year-old female with BMI 32, high cholesterol, and no chronic disease. Profile 2 is an active 45-year-old male with BMI 23, normal cholesterol, and no chronic disease. Outputs are printed as-is — no assertion on specific values is made.

In [ ]:
# Profile 1: 30yo female, BMI 32, obese — high cholesterol, sedentary, no allergies, no chronic disease
profile_obese = {
    'Age': 30, 'Gender': 'Female', 'Height_cm': 165, 'Weight_kg': 87, 'BMI': 32.0,
    'Chronic_Disease': 'None', 'Blood_Pressure_Systolic': 130, 'Blood_Pressure_Diastolic': 85,
    'Cholesterol_Level': 245, 'Blood_Sugar_Level': 100, 'Genetic_Risk_Factor': 'No',
    'Allergies': 'None', 'Daily_Steps': 3000, 'Exercise_Frequency': 1, 'Sleep_Hours': 7.0,
    'Alcohol_Consumption': 'No', 'Smoking_Habit': 'No', 'Dietary_Habits': 'Regular',
    'Caloric_Intake': 2400, 'Protein_Intake': 75, 'Carbohydrate_Intake': 290,
    'Fat_Intake': 95, 'Preferred_Cuisine': 'Western', 'Food_Aversions': 'None',
}

# Profile 2: 45yo male, BMI 23, normal — normal cholesterol, active lifestyle, no allergies, no chronic disease
profile_normal = {
    'Age': 45, 'Gender': 'Male', 'Height_cm': 178, 'Weight_kg': 73, 'BMI': 23.0,
    'Chronic_Disease': 'None', 'Blood_Pressure_Systolic': 115, 'Blood_Pressure_Diastolic': 75,
    'Cholesterol_Level': 175, 'Blood_Sugar_Level': 92, 'Genetic_Risk_Factor': 'No',
    'Allergies': 'None', 'Daily_Steps': 10000, 'Exercise_Frequency': 5, 'Sleep_Hours': 7.5,
    'Alcohol_Consumption': 'No', 'Smoking_Habit': 'No', 'Dietary_Habits': 'Regular',
    'Caloric_Intake': 2200, 'Protein_Intake': 120, 'Carbohydrate_Intake': 240,
    'Fat_Intake': 65, 'Preferred_Cuisine': 'Western', 'Food_Aversions': 'None',
}

nutrition_1, meal_plan_1 = recommend_diet(profile_obese,  obesity_class='obese')
nutrition_2, meal_plan_2 = recommend_diet(profile_normal, obesity_class='normal')

print('Profile 1 — 30yo female, BMI 32, obese, high cholesterol, sedentary:')
for k, v in nutrition_1.items():
    print(f'  {k}: {v}')
print(f'  Meal plan: {meal_plan_1}')

print()
print('Profile 2 — 45yo male, BMI 23, normal, normal cholesterol, active:')
for k, v in nutrition_2.items():
    print(f'  {k}: {v}')
print(f'  Meal plan: {meal_plan_2}')

**F.3 — Test prediction commentary.**

**Profile 1** (30yo female, BMI 32, obese, high cholesterol, sedentary): Recommended Calories 2097.2, Protein 78.9 g, Carbs 290.2 g, Fats 100.1 g. Meal plan: Low-Fat Diet.

**Profile 2** (45yo male, BMI 23, normal, normal cholesterol, active): Recommended Calories 1895.0, Protein 124.1 g, Carbs 242.4 g, Fats 69.4 g. Meal plan: High-Protein Diet.

The nutrition targets from the Linear Regression regressor show a consistent directional pattern: the obese profile receives **higher** caloric, carbohydrate, and fat recommendations than the normal-weight active profile (2097 vs 1895 kcal; 290 vs 242 g carbs; 100 vs 69 g fats), while receiving lower protein (78.9 vs 124.1 g). This pattern reflects how D4's recommendation targets were generated: the synthetic dataset appears to scale caloric and macronutrient targets with body size metrics (Weight_kg, BMI) rather than weight-management objectives. In standard clinical weight-loss guidance, an obese patient would typically receive a caloric deficit relative to their maintenance needs — the opposite of what this model outputs. This is a training-data limitation, not a model failure: Linear Regression has correctly learned the parametric relationship present in D4, but D4's generation logic does not encode clinical weight-loss intent. Phase 4 should carry a disclaimer that nutrition outputs reflect D4 training data patterns and require clinician review before use.

The meal plan predictions (Low-Fat Diet for Profile 1, High-Protein Diet for Profile 2) are not meaningfully interpretable — as established in Section E, the classifier performs at chance level (Weighted F1 0.2695) because `Recommended_Meal_Plan` in D4 has no feature correlation. The predictions are reported as produced by the model without clinical interpretation.

> **Section F complete.** Awaiting confirmation before Section G.

---
## Section G: Phase 3 summary

**Regressor comparison.** Linear Regression achieved the lowest Mean RMSE of 41.12 across the four nutrition targets, outperforming XGBoost (next best, 46.13) by 5.01 RMSE units and Random Forest (51.93) by 10.81; SVR was a distant last at 194.71, with near-zero R² for Calories (0.057) at default kernel settings (Section C results table). Linear Regression also achieved the highest Mean R² of 0.9436 — that LR outperforms ensemble methods is a direct indicator that D4's nutrition targets are near-linear synthetic functions of the input features, and the high R² values reflect parametric recovery of the generating rule, not real-world predictive skill; the synthetic-data caveat from Phase 1 EDA (|skew| < 0.02 on all four targets) is confirmed here (Section C commentary).

**Selected regressor.** Linear Regression (Pipeline: StandardScaler → LinearRegression); Mean RMSE 41.12, Mean R² 0.9436 across Recommended_Calories, Recommended_Protein, Recommended_Carbs, Recommended_Fats on the 1000-row held-out test set. Saved to `models/recommend_regressor.pkl`; round-trip verification passed (Section C code cell).

**Classifier comparison.** All four classifiers performed at chance level: Weighted F1 values ranged from 0.221 (Random Forest) to 0.270 (Logistic Regression) on a balanced 4-class problem where random guessing produces 0.25; no model achieved meaningfully above-chance classification of meal plan type (Section E results table). This is a finding about D4, not a modelling failure — cross-tabulation of `Recommended_Meal_Plan` against every feature grouping (obesity class, dietary habits, chronic disease) confirmed near-uniform ~25% distributions, indicating the meal plan target was drawn approximately independently from patient features during synthetic data generation (Section E commentary).

**Selected classifier.** Logistic Regression (Pipeline: StandardScaler → LogisticRegression) by Weighted F1 0.2695 — the highest among the four candidates by a 1.4 pp margin over SVC (0.2681); Specificity_macro 0.7576. Saved to `models/recommend_classifier.pkl`; round-trip verification passed (Section E code cell).

**Confusion matrix.** The 4×4 confusion matrix for Logistic Regression on the test set shows near-uniform spread across all predicted classes for every true class — no diagonal cell dominates, and off-diagonal cells are approximately equal in count. This is the expected visual signature of a classifier with no learned decision boundary; it confirms the cross-tabulation finding visually (Section E confusion matrix plot and commentary).

**Feature importance.** Top features by mean absolute LR coefficient across the 4 meal plan classes are BMI (0.0785), Weight_kg (0.0659), and Height_cm (0.0503) — all demographic variables — with all values close to zero in absolute terms. Even the intuitively relevant dietary features (Dietary_Habits_Vegan, Dietary_Habits_Keto) appear in the top 15 but cannot lift performance above chance, confirming that LR found no usable signal for meal plan classification in D4 regardless of feature category (Section E feature importance plot).

**Wrapper test predictions.** Both profiles ran through `recommend_diet` without error. Nutrition targets show a body-size-driven pattern: the obese profile (BMI 32) receives higher calories (2097 vs 1895), higher carbs (290 vs 242 g), and higher fats (100 vs 69 g) than the normal-weight active profile, while receiving lower protein (78.9 vs 124.1 g) — this is consistent with D4 scaling macronutrient targets to body size rather than encoding weight-loss intent, a training-data limitation that must be disclosed in Phase 4 (Section F commentary). Meal plan outputs (Low-Fat Diet and High-Protein Diet respectively) are unreliable per the classification finding and should not be presented as clinically meaningful in Phase 4.

**Documented deviations.** (1) SVR wrapped in a `MultiOutputRegressor(Pipeline([StandardScaler, SVR]))` — the proposal specifies SVR as a candidate but not a pipeline; `StandardScaler` was added because SVR with an RBF kernel uses Euclidean distances and requires feature scaling for a fair comparison against scale-invariant tree models; the pipeline is placed inside the wrapper so the saved `.pkl` handles raw input (Sections B header and B.4 markdown). (2) SVC wrapped in a `Pipeline([StandardScaler, SVC])` — same rationale as Phase 2; without scaling, unscaled features distort the RBF kernel (Sections D header and D.5 markdown).

---
## Section H: Replacing the regressor with the Mifflin-St Jeor equation

**Motivation.**  
During Phase 4 development, the saved `recommend_regressor.pkl` (Linear Regression, Mean R² 0.9436 on the D4 test set) was found to produce **physiologically invalid outputs on real patient inputs** — including negative calorie recommendations — for inputs outside the synthetic training distribution. For example, a 22-year-old female, 61 kg, 173 cm, moderate activity produced a caloric recommendation of approximately −300 kcal. Section C commentary already documented the core issue: D4 is synthetically generated, and the high R² reflects parametric recovery of the data-generating rule, not generalisation to real patients. The failure on real inputs is a direct consequence of linear extrapolation outside the narrow synthetic distribution.

**Decision.**  
The regressor is replaced with the **Mifflin-St Jeor equation** for BMR estimation, adjusted for activity level (TDEE) and weight-management intent (obesity class). Macro targets follow standard clinical proportions. This change:
- Eliminates the possibility of invalid outputs for any physiologically plausible input
- Grounds recommendations in a validated clinical formula used as standard practice by dietitians
- Removes the D4 synthetic-data limitation from the nutrition target outputs entirely

The trained `recommend_regressor.pkl` remains saved to `models/` as a record of the modelling comparison; it is no longer called at inference. The meal plan output and `recommend_classifier.pkl` are unchanged.

**Formula.**  
Mifflin-St Jeor BMR (Harris & Benedict 1918, revised Mifflin et al. 1990):  
- Female: BMR = 10 × weight_kg + 6.25 × height_cm − 5 × age − 161  
- Male:   BMR = 10 × weight_kg + 6.25 × height_cm − 5 × age + 5  

TDEE = BMR × physical activity level (PAL) multiplier (Ainsworth et al. standard values).  
Target calories = TDEE ± adjustment for weight-management intent by obesity class.  
Macros follow clinical proportions by obesity class (protein per kg body weight; fat as % of target calories; carbohydrates as remainder).

**H.1 — Clinical constants.**  
PAL multipliers follow the Ainsworth et al. (2011) compendium standard values, keyed by exercise days per week (the `Exercise_Frequency` field in D4 and the Phase 4 form). Calorie adjustments encode weight-management intent: a caloric surplus for underweight patients and a deficit for overweight/obese, consistent with Australian Dietary Guidelines recommendations. Protein targets follow the Recommended Dietary Allowance for weight management by obesity class (Leidy et al. 2015). Fat proportion follows the Acceptable Macronutrient Distribution Range (AMDR; 20–35% of calories). Carbohydrates fill the remainder. A hard floor of 1200 kcal ensures the formula never produces a physiologically unsafe recommendation regardless of input combination.

In [ ]:
# ── H.1: Clinical constants ───────────────────────────────────────────────────

# Physical activity level (PAL) multipliers keyed by exercise days/week.
# Source: Ainsworth et al. (2011) Compendium of Physical Activities.
PAL = {
    0: 1.200,   # sedentary — little or no exercise
    1: 1.375,   # lightly active — 1–2 days/week
    2: 1.375,
    3: 1.550,   # moderately active — 3–4 days/week
    4: 1.550,
    5: 1.725,   # very active — 5–6 days/week
    6: 1.725,
    7: 1.900,   # extra active — daily vigorous exercise
}

# Calorie adjustment (kcal/day) relative to TDEE by obesity class.
CALORIE_ADJUST = {
    'underweight': +300,   # anabolic surplus to support weight gain
    'normal':          0,  # maintenance
    'overweight':   -300,  # mild deficit (~0.3 kg/week loss)
    'obese':        -500,  # moderate deficit (~0.5 kg/week loss)
}

# Protein target (g per kg body weight) by obesity class.
# Source: Leidy et al. (2015).
PROTEIN_G_PER_KG = {
    'underweight': 1.6,
    'normal':      1.2,
    'overweight':  1.4,
    'obese':       1.6,
}

# Additional protein cap: protein calories never exceed 30% of target calories.
# Prevents unrealistically high absolute protein amounts for very heavy patients
# (e.g. 200 kg obese patient at 1.6 g/kg = 320 g = 43% of calories without the cap).
# 30% is the upper bound of the Acceptable Macronutrient Distribution Range (AMDR).
PROTEIN_CAL_FRACTION_MAX = 0.30

# Fat as fraction of target calories (within AMDR 20–35%) by obesity class.
FAT_FRACTION = {
    'underweight': 0.30,
    'normal':      0.30,
    'overweight':  0.28,
    'obese':       0.25,
}

# Hard floor: minimum safe caloric intake regardless of formula output.
CALORIE_FLOOR = 1200.0

print('Clinical constants defined.')
print(f'PAL range: {min(PAL.values())} – {max(PAL.values())}')
print(f'Calorie adjustments: {CALORIE_ADJUST}')
print(f'Protein targets (g/kg): {PROTEIN_G_PER_KG}')
print(f'Protein calorie cap: ≤ {PROTEIN_CAL_FRACTION_MAX*100:.0f}% of target calories')
print(f'Fat fractions: {FAT_FRACTION}')
print(f'Calorie floor: {CALORIE_FLOOR} kcal')

**H.2 — `mifflin_st_jeor_nutrition` function.**  
Implements the complete pipeline: Mifflin-St Jeor BMR → TDEE → target calories → macros. The function signature mirrors `recommend_nutrition` in `pipeline/recommend.py` exactly, making it a drop-in replacement. The 1200 kcal floor is applied after the obesity-class adjustment; in practice it only activates for very low body weight combined with a large deficit (e.g. an underweight patient incorrectly classified as obese), protecting against edge-case extrapolation.

In [ ]:
def mifflin_st_jeor_nutrition(clinical_dict: dict, obesity_class: str) -> dict:
    """
    Calculate recommended daily nutrition targets using the Mifflin-St Jeor
    equation (BMR), adjusted for activity level (TDEE) and obesity class.

    Parameters
    ----------
    clinical_dict : dict
        Must contain Age (int/float), Gender (str 'Female'/'Male'),
        Height_cm (float), Weight_kg (float),
        Exercise_Frequency (int, days/week 0–7).
    obesity_class : str
        One of 'underweight', 'normal', 'overweight', 'obese'.

    Returns
    -------
    dict with keys Recommended_Calories, Recommended_Protein,
    Recommended_Carbs, Recommended_Fats (all float, rounded to 1 dp).
    """
    age     = float(clinical_dict['Age'])
    gender  = str(clinical_dict['Gender']).strip().lower()
    height  = float(clinical_dict['Height_cm'])
    weight  = float(clinical_dict['Weight_kg'])
    ex_freq = int(clinical_dict.get('Exercise_Frequency', 3))
    oc      = obesity_class.lower()

    # Step 1: Mifflin-St Jeor BMR
    if gender == 'female':
        bmr = 10 * weight + 6.25 * height - 5 * age - 161
    else:
        bmr = 10 * weight + 6.25 * height - 5 * age + 5

    # Step 2: TDEE = BMR × activity multiplier
    pal  = PAL.get(min(max(ex_freq, 0), 7), 1.55)
    tdee = bmr * pal

    # Step 3: Target calories = TDEE ± weight-management adjustment, floored at 1200 kcal
    target_cal = max(tdee + CALORIE_ADJUST.get(oc, 0), CALORIE_FLOOR)

    # Step 4: Protein — body-weight scaled, capped at 30% of target calories.
    # The per-kg target can produce unrealistically high absolute amounts for very
    # heavy patients (e.g. 200 kg × 1.6 g/kg = 320 g = 43% of calories). The 30%
    # AMDR ceiling ensures protein stays within accepted clinical bounds regardless
    # of body weight.
    protein_g = min(
        PROTEIN_G_PER_KG.get(oc, 1.2) * weight,
        PROTEIN_CAL_FRACTION_MAX * target_cal / 4,
    )

    # Step 5: Fat — percentage of target calories (within AMDR 20–35%)
    fat_g = (FAT_FRACTION.get(oc, 0.30) * target_cal) / 9

    # Step 6: Carbs — remainder after protein and fat calories
    # With protein ≤ 30% and fat ≤ 30%, carbs are always ≥ 40% of target,
    # so the zero floor here is a safety net only.
    carb_g = max((target_cal - protein_g * 4 - fat_g * 9) / 4, 0.0)

    return {
        'Recommended_Calories': round(target_cal, 1),
        'Recommended_Protein':  round(protein_g,  1),
        'Recommended_Carbs':    round(carb_g,      1),
        'Recommended_Fats':     round(fat_g,       1),
    }

print('mifflin_st_jeor_nutrition defined.')

**H.3 — Validation across representative profiles.**  
Six profiles are tested spanning the full range of inputs: the original failing case (22F, 61 kg, normal BMI) that produced −300 kcal from the regressor; the three Phase 4 sample patients; a sedentary obese male; and an elderly underweight female. Each output is checked against physiological plausibility criteria: calories between 1200–4000 kcal, protein between 40–200 g, carbs ≥ 0 g, fats between 20–150 g. Directional expectations are also verified: obese profiles should receive a caloric deficit relative to TDEE; underweight profiles a surplus; heavier/taller patients should receive more calories than lighter/shorter patients at the same activity level.

In [ ]:
VALIDATION_PROFILES = [
    # (label, clinical_dict, obesity_class, notes)
    (
        'Original failing case — 22F, 61 kg, 173 cm, normal, moderate activity',
        {'Age': 22, 'Gender': 'Female', 'Height_cm': 173.0, 'Weight_kg': 61.0, 'Exercise_Frequency': 3},
        'normal',
        'Regressor produced ~−300 kcal; formula must return positive value ≥ 1200',
    ),
    (
        'Phase 4 sample: High-risk — 52F, 95 kg, 165 cm, obese, 1 day exercise',
        {'Age': 52, 'Gender': 'Female', 'Height_cm': 165.0, 'Weight_kg': 95.0, 'Exercise_Frequency': 1},
        'obese',
        'Obese → caloric deficit; calories should be TDEE − 500',
    ),
    (
        'Phase 4 sample: Moderate — 38M, 88 kg, 178 cm, overweight, 3 days',
        {'Age': 38, 'Gender': 'Male', 'Height_cm': 178.0, 'Weight_kg': 88.0, 'Exercise_Frequency': 3},
        'overweight',
        'Overweight → mild deficit; calories should be TDEE − 300',
    ),
    (
        'Phase 4 sample: Low-risk — 28F, 62 kg, 170 cm, normal, 5 days',
        {'Age': 28, 'Gender': 'Female', 'Height_cm': 170.0, 'Weight_kg': 62.0, 'Exercise_Frequency': 5},
        'normal',
        'Normal weight → maintenance; calories ≈ TDEE',
    ),
    (
        'Sedentary obese male — 45M, 120 kg, 175 cm, obese, 0 days',
        {'Age': 45, 'Gender': 'Male', 'Height_cm': 175.0, 'Weight_kg': 120.0, 'Exercise_Frequency': 0},
        'obese',
        'Large body mass, sedentary; should still produce plausible positive calorie target',
    ),
    (
        'Elderly underweight female — 75F, 48 kg, 158 cm, underweight, 1 day',
        {'Age': 75, 'Gender': 'Female', 'Height_cm': 158.0, 'Weight_kg': 48.0, 'Exercise_Frequency': 1},
        'underweight',
        'Underweight → caloric surplus; calories should be TDEE + 300',
    ),
    (
        'Extreme: max weight sedentary obese — 45M, 200 kg, 175 cm, obese, 0 days',
        {'Age': 45, 'Gender': 'Male', 'Height_cm': 175.0, 'Weight_kg': 200.0, 'Exercise_Frequency': 0},
        'obese',
        'Without protein cap: 320 g protein = 43% of calories. Cap must bring protein ≤ 30%.',
    ),
    (
        'Extreme: min age+weight floor case — 90F, 40 kg, 140 cm, obese, 0 days',
        {'Age': 90, 'Gender': 'Female', 'Height_cm': 140.0, 'Weight_kg': 40.0, 'Exercise_Frequency': 0},
        'obese',
        'Very low TDEE; 1200 kcal floor must activate without distorting macros',
    ),
]

print('=' * 85)
all_pass = True
for label, cdict, oc, notes in VALIDATION_PROFILES:
    result = mifflin_st_jeor_nutrition(cdict, oc)
    cal  = result['Recommended_Calories']
    prot = result['Recommended_Protein']
    carb = result['Recommended_Carbs']
    fat  = result['Recommended_Fats']

    # Plausibility checks
    checks = {
        'cal ≥ 1200':           cal  >= 1200,
        'cal ≤ 7000':           cal  <= 7000,
        'prot ≥ 40':            prot >= 40,
        'prot ≤ 30% cal':       prot * 4 <= cal * 0.301,   # 0.001 tolerance for rounding
        'carb ≥ 0':             carb >= 0,
        'fat ≥ 15':             fat  >= 15,
        'macros sum ≈ cal':     abs((prot*4 + fat*9 + carb*4) - cal) < 2,
    }
    passed = all(checks.values())
    all_pass = all_pass and passed
    status = '✓ PASS' if passed else '✗ FAIL'

    print(f'\n{status}  {label}')
    print(f'        Note: {notes}')
    print(f'        Cal: {cal} kcal | Prot: {prot} g ({prot*4/cal*100:.0f}%) | '
          f'Carb: {carb} g | Fat: {fat} g')
    if not passed:
        failed = [k for k, v in checks.items() if not v]
        print(f'        FAILED CHECKS: {failed}')

print('\n' + '=' * 85)
print(f'\nOverall: {"ALL PROFILES PASS" if all_pass else "FAILURES DETECTED — review above"}')

**H.3 — Validation commentary.**

All eight profiles pass all plausibility checks. Key results:

| Profile | Calories | Protein (% cal) | Carbs | Fats |
|---|---|---|---|---|
| 22F 61 kg normal (original failing case) | 2,201 kcal | 73.2 g (13%) | 312.0 g | 73.4 g |
| 52F 95 kg obese (high-risk sample) | 1,645 kcal | 123.4 g (30%) | 185.1 g | 45.7 g |
| 38M 88 kg overweight (moderate sample) | 2,502 kcal | 123.2 g (20%) | 327.1 g | 77.8 g |
| 28F 62 kg normal (low-risk sample) | 2,383 kcal | 74.4 g (12%) | 342.6 g | 79.4 g |
| 45M 120 kg obese sedentary | 1,989 kcal | 192.0 g (39%) → capped | — | — |
| **45M 200 kg obese sedentary (extreme)** | **2,949 kcal** | **221.1 g (30%)** | **331.7 g** | **81.9 g** |
| 90F 40 kg obese (1200 kcal floor) | 1,200 kcal | 64.0 g (21%) | 161.0 g | 33.3 g |
| 75F 48 kg underweight | 1,581 kcal | 76.8 g (19%) | 199.8 g | 52.7 g |

**Protein cap in action.** The 200 kg obese patient demonstrates the cap: the per-kg formula prescribes 320 g protein (43% of calories), which the AMDR ceiling reduces to 221 g (30%). Without the cap this patient would have received a clinically extreme protein load; with it, the recommendation falls within accepted guidelines. The 95 kg obese patient also hits the cap at exactly 30% (123.4 g), because the prescribed 152 g at 1.6 g/kg exceeds the ceiling.

**Directional correctness.** Obese profiles receive TDEE − 500 kcal; overweight receives TDEE − 300 kcal; normal-weight receives maintenance; underweight receives TDEE + 300 kcal. The 1200 kcal floor activates for the elderly low-BMR case without distorting the macro split.

**The original failing case is resolved.** The 22F, 61 kg, 173 cm profile that produced approximately −300 kcal from the Linear Regression regressor now correctly returns 2,201 kcal.

**Phase 4 deployment.** The `mifflin_st_jeor_nutrition` function is deployed in `pipeline/recommend.py` with the same protein cap and floors. The `recommend_classifier.pkl` and meal plan logic are unchanged.